In [3]:
import pandas as pd
from dotenv import load_dotenv
import os
import time
from openai import OpenAI


In [4]:
data_high = pd.read_json("../data/llm_formatted/high_llm.json")
data_low = pd.read_json("../data/llm_formatted/low_llm.json")
TARGET = "HubSpot"

In [5]:
data = pd.concat([data_high, data_low], axis=0)
data = data.reset_index(drop=True)


In [6]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [7]:
import json

def llm_sentiment(thread, target, subreddit):
    prompt = f'''
        You are analyzing Reddit threads about companies.

        Target company: {target}
        Subreddit: {subreddit}
        Thread: {thread}

        Your task is to classify the overall sentiment of the thread toward {target}.

        If the thread is from a subreddit dedicated to the company, assume it is about {target}, even if the name is not directly mentioned.
        Focus on what people in the thread actually think about {target}, not just the general tone.

        Classify the overall sentiment of the thread toward {target} as:
        - "positive"
        - "negative"
        - "neutral"

        Use these guidelines:
        - "positive" - users are clearly satisfied, happy, or say something good about {target}
        - "negative" - users complain, express frustration, describe problems, or prefer alternatives over {target}
        - "neutral" - the thread mainly asks questions, shares information, or has no clear opinion

        Questions are usually neutral, unless they clearly describe a problem or frustration related to {target}
        A positive or enthusiastic tone alone does not necessarily mean positive sentiment.

        If opinions differ, use the dominant sentiment towards {target} across the full thread.
        If there is no clear opinion about {target}, choose "neutral".

        Return only this JSON:
        {{
            "sentiment": "positive" | "negative" | "neutral"
        }}
    '''

    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
        )

        raw = response.output_text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        analysis = json.loads(raw)
        return analysis

    except json.JSONDecodeError:
        print("JSON error on:", response.output_text)
        return {
            "sentiment": "neutral"
        }

In [8]:
def llm_analysis(thread, target, subreddit):
    prompt = f'''
        You are analyzing Reddit threads about companies.

        Target company: {target}
        Subreddit: {subreddit}
        Thread: {thread}

        Your task is to decide whether this thread contains useful product feedback about {target}.

        If the thread is from a subreddit dedicated to the company, assume it is about {target}, even if the name is not directly mentioned.
        Focus on parts of the thread that provide meaningful feedback about {target}, such as problems, limitations, needs, comparisons, or concrete suggestions.

        1. Decide whether the thread contains actionable product feedback.
        Set "product_feedback" to true only if the thread contains a clear problem, limitation, complaint, suggestion, or concrete request related to {target}.
        Otherwise set it to false.

        2. Provide a relevancy score from 0 to 100 that reflects how useful this thread is for understanding user feedback about {target}.
        Higher score = more useful, more specific, more actionable for a product manager.

        Use this scale:
        - 0-20: irrelevant or no useful information about {target}
        - 21-40: mentions {target}, but there's no real feedback or only vague opinions
        - 41-60: somewhat useful feedback or question
        - 61-80: clear feedback related to {target}
        - 81-100: very specific, actionable product feedback with a strong value for product manager

        3. Identify all relevant labels from the list below. You may return multiple labels or an empty list [].
        Only assign a label if there is clear evidence in the thread.

        Labels:
        - "bug_report": something is broken or not working
        - "limitation": works but lacks features or does not meet needs
        - "comparison": mentions competitors or alternative tools
        - "churn_risk": user is considering leaving or has already switched away
        - "purchase_intent": user is considering starting to use the product

        If the thread clearly contains another useful category that is not covered by the list above, include it in "additional_labels".
        Only include them if they are clearly supported by the thread.

        4. Extract 1-3 key highlights from the thread that capture the most important opinions, complaints, or praise about {target}.

        Each highlight must:
        - contain a short exact quote from the original thread. Do not paraphrase
        - include the main product aspect it relates to

        Use one aspect per highlight from this list:
        - "pricing"
        - "usability"
        - "performance"
        - "integration"
        - "automation"
        - "reporting"
        - "API"
        - "AI"
        - "onboarding"
        - "support"

        If none of these fit well, you may use a short custom aspect (1-3 words maximum).

        5. Write a short 1-2 sentence summary of the thread in Estonian.

        Return only this JSON:
        {{
            "product_feedback": true | false,
            "relevancy_score": 0,
            "labels": [],
            "additional_labels": [],
            "highlights": [
                {{
                    "quote": "",
                    "aspect": ""
                }}
            ],
            "summary_et": "lühike kokkuvõte eesti keeles"
        }}
'''

    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
        )

        raw = response.output_text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        analysis = json.loads(raw)
        return analysis

    except json.JSONDecodeError:
        print("JSON error on:", response.output_text)
        return {
            "product_feedback": False,
            "relevancy_score": 0,
            "labels": [],
            "additional_labels": [],
            "highlights": [],
            "summary_et": "Viga JSON-vastuse töötlemisel."
        }

In [9]:
def run_sentiment(data):
    rows = []
    start_time = time.time()

    for i, row in data.iterrows():
        thread = row["thread_for_llm"]
        subreddit = row["subreddit_name"]

        out = {
            "id": row.get("id"),
            "thread_for_llm": thread
        }

        try:
            sentiment_result = llm_sentiment(thread, TARGET, subreddit)

            out.update({
                "llm_sentiment": sentiment_result.get("sentiment", "neutral")
            })

        except Exception as e:
            out.update({
                "llm_sentiment": "neutral"
            })

        rows.append(out)

        if (i + 1) % 50 == 0 or (i + 1) == len(data):
            print(i+1)

    print(f"Time: {time.time() - start_time:.2f}s")
    return pd.DataFrame(rows)

In [1]:
def run_feedback(data):
    rows = []
    start_time = time.time()

    for i, row in data.iterrows():
        thread = row["thread_for_llm"]
        subreddit = row["subreddit_name"]

        out = {
            "id": row.get("id"),
            "thread_for_llm": thread
        }

        try:
            feedback_result = llm_analysis(thread, TARGET, subreddit)

            out.update({
                "product_feedback": feedback_result.get("product_feedback", False),
                "relevancy_score": feedback_result.get("relevancy_score", 0),
                "labels": feedback_result.get("labels", []),
                "additional_labels": feedback_result.get("additional_labels", []),
                "highlights": feedback_result.get("highlights", []),
                "summary_et": feedback_result.get("summary_et", "")
            })

        except Exception as e:
            out.update({
                "product_feedback": False,
                "relevancy_score": 0,
                "labels": [],
                "additional_labels": [],
                "highlights": [],
                "summary_et": f"ERROR: {e}"
            })

        rows.append(out)

        if (i + 1) % 50 == 0 or (i + 1) == len(data):
            print({i+1})
            print(out)

    print(f"Time: {time.time() - start_time:.2f}s")
    return pd.DataFrame(rows)

In [10]:
print("Sentiment")
sentiment_data = run_sentiment(data)
print("Analysis")
feedback_data = run_feedback(data)

results = sentiment_data.merge(feedback_data, on=["id", "thread_for_llm"], how="left")

Sentiment
50
100
150
200
250
300
350
400
450
500
550
600
650
700
750
800
850
900
950
1000
1050
1100
1150
1200
1250
1266
Time: 1364.58s
Analysis
JSON error on: {
  "product_feedback": true,
  "relevancy_score": 75,
  "labels": ["limitation"],
  "additional_labels": [],
  "highlights": [
    {
      "quote": "it's all about killing that repetitive micro-friction",
      "aspect": "usability"
    },
    {
      "quote": "the extension intercepts it: it opens Google Voice to dial the number AND automatically triggers HubSpot's native call logging panel for that specific contact",
      "aspect": "integration"
    }
  ],
  "summary_et": "Kasutaja jagab tasuta Chrome\'i laiendust, mis hõlbustab HubSpoti ja Google Voice'i kooskasutamist, parandades helistamise protsessi ning automaatse kõnelogi sisselülitamise võimalust HubSpotis, vähendades käsitsi andmete kopeerimist."
}
{50}
{'id': '1s8bv0v', 'thread_for_llm': 'SUBREDDIT: hubspot\n\nPOST [post_id_1s8bv0v]\nTITLE: Third client this month wh

In [11]:
os.makedirs("../data/thread_level_results", exist_ok=True)
results.to_json("../data/thread_level_results/thread_results.json", orient="records", indent=2, force_ascii=False)
results.to_csv("../data/thread_level_results/thread_results.csv",index=False,encoding="utf-8")

## Jooksutan Json parsimisega halvasti läinud vastused uuesti

In [17]:
results[results["summary_et"] == "Viga JSON-vastuse töötlemisel."]

,id,thread_for_llm,llm_sentiment,product_feedback,relevancy_score,labels,additional_labels,highlights,summary_et
37,1samxgy,SUBREDDIT: hubspot\n\nPOST [post_id_1samxgy]\n...,positive,False,0,[],[],[],Viga JSON-vastuse töötlemisel.
536,1q8p757,SUBREDDIT: hubspot\n\nPOST [post_id_1q8p757]\n...,negative,False,0,[],[],[],Viga JSON-vastuse töötlemisel.
763,1p5clfl,SUBREDDIT: hubspot\n\nPOST [post_id_1p5clfl]\n...,neutral,False,0,[],[],[],Viga JSON-vastuse töötlemisel.
974,1o7issn,SUBREDDIT: hubspot\n\nPOST [post_id_1o7issn]\n...,neutral,False,0,[],[],[],Viga JSON-vastuse töötlemisel.
996,1owd89h,SUBREDDIT: marketing\n\nPOST [post_id_1owd89h]...,positive,False,0,[],[],[],Viga JSON-vastuse töötlemisel.
1025,1s1b8ct,SUBREDDIT: CRM\n\nPOST [post_id_1s1b8ct]\nTITL...,negative,False,0,[],[],[],Viga JSON-vastuse töötlemisel.


In [18]:
import re
import json

In [19]:
def llm_analysis_new(thread, target, subreddit):
    prompt = f'''
        You are analyzing Reddit threads about companies.

        Target company: {target}
        Subreddit: {subreddit}
        Thread: {thread}

        Your task is to decide whether this thread contains useful product feedback about {target}.

        If the thread is from a subreddit dedicated to the company, assume it is about {target}, even if the name is not directly mentioned.
        Focus on parts of the thread that provide meaningful feedback about {target}, such as problems, limitations, needs, comparisons, or concrete suggestions.

        1. Decide whether the thread contains actionable product feedback.
        Set "product_feedback" to true only if the thread contains a clear problem, limitation, complaint, suggestion, or concrete request related to {target}.
        Otherwise set it to false.

        2. Provide a relevancy score from 0 to 100 that reflects how useful this thread is for understanding user feedback about {target}.
        Higher score = more useful, more specific, more actionable for a product manager.

        Use this scale:
        - 0-20: irrelevant or no useful information about {target}
        - 21-40: mentions {target}, but there's no real feedback or only vague opinions
        - 41-60: somewhat useful feedback or question
        - 61-80: clear feedback related to {target}
        - 81-100: very specific, actionable product feedback with a strong value for product manager

        3. Identify all relevant labels from the list below. You may return multiple labels or an empty list [].
        Only assign a label if there is clear evidence in the thread.

        Labels:
        - "bug_report": something is broken or not working
        - "limitation": works but lacks features or does not meet needs
        - "comparison": mentions competitors or alternative tools
        - "churn_risk": user is considering leaving or has already switched away
        - "purchase_intent": user is considering starting to use the product

        If the thread clearly contains another useful category that is not covered by the list above, include it in "additional_labels".
        Only include them if they are clearly supported by the thread.

        4. Extract 1-3 key highlights from the thread that capture the most important opinions, complaints, or praise about {target}.

        Each highlight must:
        - contain a short exact quote from the original thread. Do not paraphrase
        - include the main product aspect it relates to

        Use one aspect per highlight from this list:
        - "pricing"
        - "usability"
        - "performance"
        - "integration"
        - "automation"
        - "reporting"
        - "API"
        - "AI"
        - "onboarding"
        - "support"

        If none of these fit well, you may use a short custom aspect (1-3 words maximum).

        5. Write a short 1-2 sentence summary of the thread in Estonian.

        Return only this JSON:
        {{
            "product_feedback": true | false,
            "relevancy_score": 0,
            "labels": [],
            "additional_labels": [],
            "highlights": [
                {{
                    "quote": "",
                    "aspect": ""
                }}
            ],
            "summary_et": "lühike kokkuvõte eesti keeles"
        }}
'''

    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
        )

        raw = response.output_text.strip()

        try:
            text = raw

            text = re.sub(r"^```json\s*", "", text)
            text = re.sub(r"^```\s*", "", text)
            text = re.sub(r"\s*```$", "", text)

            match = re.search(r"\{.*\}", text, re.DOTALL)
            if not match:
                raise ValueError("No JSON found")

            analysis = json.loads(match.group(0))
            return analysis

        except Exception as e:
            return {
                "product_feedback": False,
                "relevancy_score": 0,
                "labels": [],
                "additional_labels": [],
                "highlights": [],
                "summary_et": f"Viga JSON-vastuse töötlemisel.",
                "raw_response": raw
            }

    except Exception as e:
        return {
            "product_feedback": False,
            "relevancy_score": 0,
            "labels": [],
            "additional_labels": [],
            "highlights": [],
            "summary_et": f"ERROR: {e}"
        }

In [20]:
import pandas as pd
import os

bad = results[results["summary_et"] == "Viga JSON-vastuse töötlemisel."].copy()
bad[["id", "llm_sentiment", "product_feedback", "relevancy_score", "summary_et"]]

,id,llm_sentiment,product_feedback,relevancy_score,summary_et
37,1samxgy,positive,False,0,Viga JSON-vastuse töötlemisel.
536,1q8p757,negative,False,0,Viga JSON-vastuse töötlemisel.
763,1p5clfl,neutral,False,0,Viga JSON-vastuse töötlemisel.
974,1o7issn,neutral,False,0,Viga JSON-vastuse töötlemisel.
996,1owd89h,positive,False,0,Viga JSON-vastuse töötlemisel.
1025,1s1b8ct,negative,False,0,Viga JSON-vastuse töötlemisel.


In [21]:
fixed_rows = []

for n, (i, row) in enumerate(bad.iterrows()):
    out = row.to_dict()

    try:
        feedback_result = llm_analysis_new(
            row["thread_for_llm"],
            TARGET,
            row.get("subreddit_name", "")
        )

        out.update({
            "product_feedback": feedback_result.get("product_feedback", False),
            "relevancy_score": feedback_result.get("relevancy_score", 0),
            "labels": feedback_result.get("labels", []),
            "additional_labels": feedback_result.get("additional_labels", []),
            "highlights": feedback_result.get("highlights", []),
            "summary_et": feedback_result.get("summary_et", "")
        })

    except Exception as e:
            out.update({
                "product_feedback": False,
                "relevancy_score": 0,
                "labels": [],
                "additional_labels": [],
                "highlights": [],
                "summary_et": f"ERROR: {e}"
            })

    fixed_rows.append(out)
fixed = pd.DataFrame(fixed_rows)

In [23]:
fixed[fixed["summary_et"] == "Viga JSON-vastuse töötlemisel."]

,id,thread_for_llm,llm_sentiment,product_feedback,relevancy_score,labels,additional_labels,highlights,summary_et


In [24]:
columns = ["product_feedback", "relevancy_score", "labels", "additional_labels", "highlights", "summary_et"]
results_fixed = results.copy().set_index("id")
fixed_indexed = fixed.set_index("id")

results_fixed.loc[fixed_indexed.index, columns] = fixed_indexed[columns]
results_fixed = results_fixed.reset_index()

In [25]:
os.makedirs("../data/thread_level_results", exist_ok=True)
results_fixed.to_json("../data/thread_level_results/thread_results_fixed.json", orient="records", indent=2, force_ascii=False)
results_fixed.to_csv("../data/thread_level_results/thread_results_fixed.csv", index=False, encoding="utf-8")

## Ühendame andmed

In [26]:
import pandas as pd
import json
import os
import re

In [ ]:
def normalize_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def find_highlight_location(highlights, raw_thread, cleaned_thread):
    fixed_highlights = []

    if not isinstance(highlights, list):
        return []

    raw_norm = normalize_text(raw_thread)
    cleaned_norm = normalize_text(cleaned_thread)

    for h in highlights:
        if isinstance(h, dict):
            quote = h.get("quote", "")
            aspect = h.get("aspect", "")
        else:
            quote = str(h)
            aspect = ""

        quote_norm = normalize_text(quote)

        source = "not_found"

        if quote_norm and quote_norm in raw_norm:
            source = "raw_thread"
        elif quote_norm and quote_norm in cleaned_norm:
            source = "cleaned_thread"

        fixed_highlights.append({
            "quote": quote,
            "aspect": aspect,
            "source": source
        })

    return fixed_highlights


In [ ]:
with open("../data/raw/low.json", "r", encoding="utf-8") as f:
    raw_low = json.load(f)

with open("../data/raw/high.json", "r", encoding="utf-8") as f:
    raw_high = json.load(f)

raw_all = raw_low + raw_high

raw_rows = []

for post in raw_all:
    post_id = post.get("id")

    title = post.get("title", "") or ""
    selftext = post.get("selftext", "") or ""
    subreddit = post.get("subreddit_name", "") or ""

    raw_thread_parts = [
        f"SUBREDDIT: {subreddit}",
        "",
        f"POST [{post_id}]",
        f"TITLE: {title}",
        f"BODY: {selftext}",
        "",
        "COMMENTS:"
    ]

    for c in post.get("comments", []):
        comment_id = c.get("id", "")
        parent_id = c.get("parent_id", "")
        depth = c.get("depth", 0)
        score = c.get("score", 0)
        body = c.get("body", "") or ""

        indent = "  " * int(depth)

        raw_thread_parts.append(
            f"{indent}COMMENT [{comment_id}]\n"
            f"{indent}parent_id: {parent_id}\n"
            f"{indent}depth: {depth}\n"
            f"{indent}score: {score}\n"
            f"{indent}text: {body}"
        )

    raw_thread = "\n\n".join(raw_thread_parts)

    raw_rows.append({
        "id": post_id,
        "raw_title": title,
        "raw_selftext": selftext,
        "raw_subreddit_name": subreddit,
        "raw_score": post.get("score"),
        "raw_upvote_ratio": post.get("upvote_ratio"),
        "raw_created_utc": post.get("created_utc"),
        "raw_url": post.get("url"),
        "raw_permalink": post.get("permalink"),
        "raw_comments": post.get("comments", []),
        "raw_thread": raw_thread
    })

raw_df = pd.DataFrame(raw_rows)

In [ ]:
results = pd.read_json("../data/thread_level_results/thread_results_fixed.json")
mvp = results.merge(raw_df, on="id", how="left")


mvp["highlights_for_mvp"] = mvp.apply(
    lambda row: find_highlight_location(
        row.get("highlights", []),
        row.get("raw_thread", ""),
        row.get("thread_for_llm", "")
    ),
    axis=1
)

mvp["display_thread"] = mvp.apply(
    lambda row: row["raw_thread"] if pd.notna(row.get("raw_thread")) and row.get("raw_thread") else row.get("thread_for_llm", ""),
    axis=1
)

In [ ]:
os.makedirs("../data/thread_level_results", exist_ok=True)
mvp.to_json("../data/thread_level_results/thread_results_mvp.json",orient="records",indent=2,force_ascii=False)
mvp.to_csv("../data/thread_level_results/thread_results_mvp.csv",index=False, encoding="utf-8")

## Leiame kasutatud sildid

In [ ]:
import pandas as pd
import ast
from collections import Counter

In [ ]:
data = pd.read_csv("../data/thread_level_results/thread_results_mvp.csv")

all_labels = []

for col in ["labels", "additional_labels"]:
    for x in data[col]:
        if pd.isna(x):
            continue
        all_labels.extend(ast.literal_eval(x))

label_counts = Counter(all_labels)

for label, count in label_counts.most_common():
    print(label, count)